In [3]:
### Homework: Building a Wikipedia Agent
build an agent that can search Wikipedia and fetch page content to answer questions: defining tools -> create an agent that uses these tools to answer questions

SyntaxError: invalid syntax (2646569687.py, line 2)

In [ ]:
# Wikipedia API Setup
# two Wikipedia API endpoints
# 1. Search API - Find pages related to a topic: https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch=YOUR_QUERY
# 2. Get Page API - Fetch the raw content of a page: https://en.wikipedia.org/w/index.php?title=PAGE_TITLE&action=raw

### Question 1. Define the Search Tool
Create a search_wikipedia function that:

Takes a query string as input
Calls the Wikipedia Search API
Returns the search results
Test your function with the query "capybara".

In [4]:
# openAI set up
from openai import OpenAI
openai_client = OpenAI()

In [ ]:
# Define the search function:
import requests

def search_wikipedia(query):
    response = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "query",
            "format": "json",
            "list": "search",
            "srsearch": query,
        },
        headers={
            "User-Agent": "AgentFlowLearning/1.0 (educational notebook)"
        },
        timeout=20,
    )
    response.raise_for_status()
    return response.json()["query"]["search"]

In [ ]:
results = search_wikipedia("capybara")
print(results)

[{'ns': 0, 'title': 'Capybara', 'pageid': 6776, 'size': 44903, 'wordcount': 4312, 'snippet': 'The <span class="searchmatch">capybara</span> (/kæpɪˈbɑːrə/ kap-ih-BAR-uh) or greater <span class="searchmatch">capybara</span> (Hydrochoerus hydrochaeris) is the largest living rodent, native to all countries in South', 'timestamp': '2026-10-11T22:03:04Z'}, {'ns': 0, 'title': 'Lesser capybara', 'pageid': 23188846, 'size': 5681, 'wordcount': 582, 'snippet': 'The lesser <span class="searchmatch">capybara</span> (Hydrochoerus isthmius) is a large semi-aquatic rodent found in South America that has vast similarities with, yet subtle differences', 'timestamp': '2026-08-14T00:29:52Z'}, {'ns': 0, 'title': 'Capybara (disambiguation)', 'pageid': 69085306, 'size': 535, 'wordcount': 101, 'snippet': 'Look up <span class="searchmatch">capybara</span>\xa0or <span class="searchmatch">Capybara</span> in Wiktionary, the free dictionary. The <span class="searchmatch">capybara</span> is a giant cavy rodent nati

### Question 3. Define the Get Page Tool
Create a get_page function that:

- Takes a page title as input 
- Calls the Wikipedia Get Page API
- Returns the page content as a string 

Test your function by fetching the page titled "Capybara".

In [7]:
# Define the get_page function:
import requests

def get_page(PAGE_TITLE):
    response = requests.get(
        "https://en.wikipedia.org/w/index.php",
        params={
            "title": PAGE_TITLE,
            "action": "raw",
        },
        headers={
            "User-Agent": "AgentFlowLearning/1.0 (educational notebook)"
        },
        timeout=20,
    )
    response.raise_for_status()
    return response.text

In [ ]:
results =get_page("Capybara")

# counts characters including spaces, tabs (\t), and newlines (\n).
len(results)

44767

### Question 4. Agent Setup
Now create an agent with access to both tools:

search_wikipedia - Search for Wikipedia pages
get_page - Fetch the content of a specific page
You can use any framework (ToyAIKit, PydanticAI, OpenAI Agents SDK, LangChain, etc.) or implement the agentic loop yourself from scratch.

In your submission, please specify:

Which framework did you choose (or "custom" if implementing from scratch)?
Which LLM provider did you use?
This question is for tracking purposes - any reasonable choice is correct.

PydanticAI + OpenAI

### Question 5. Testing Your Agent - Single Page
Use your agent to answer this question:

"What is this page about? https://en.wikipedia.org/wiki/Capybara"

Your agent should:

Use the get_page tool to fetch the Capybara page
Provide a concise summary of what the page is about
What is the main topic of the Capybara page according to your agent?

Keep the answer ready for submission.

1. build prompt 
agentic -> question are turned to arguments ??? extract the PAGE_TITLE from link in the question 
get_page -> use PAGE_TITLE to call get_page tool and return the response.text -> this will be the context
form the prompt

Traditional get_page agent:
PAGE_TITLE → get page → LLM → Answer

Agentic get_page agent:
Question → LLM chooses PAGE_TITLE → get page → LLM → Answer


2. Get the response from the LLM


In [6]:
# set up the prompt template:
import json

PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, page_results):
    context = json.dumps(page_results, indent=2)
    return PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

In [17]:
# Define the get_page tool schema:

instructions = """
You're a Wikipedia assistant. 

Answer the user question using the Wikipedia page knowledge base. Provide a concise summary of what the page is about.

Use only facts from the knowledge base when answering.
IMPORTANT: f you cannot find the answer, inform the user.
"""


get_page_tool = {
    "type": "function",
    "name": "get_page",
    "description": "Fetch the raw content of a Wikipedia page for relevant results based on a PAGE_TITLE string.",
    "parameters": {
        "type": "object",
        "properties": {
            "PAGE_TITLE": {
                "type": "string",
                "description": "The page title to look up in the Wikipedia Get Page API endpoints"
            }
        },
        "required": [
            "PAGE_TITLE"
        ]
    }
}

In [22]:
import json

# Make a query and build the prompt:
question = "What is this page about? https://en.wikipedia.org/wiki/Capybara"

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

# 1. Ask the LLM to generate the get_page arguments.
response = openai_client.responses.create(
    model='gpt-6-luna',
    input=messages,
    tools=[get_page_tool],
)

# 2. Find the function call; output[0] could be another item type.
tool_call = next(
    (item for item in response.output if item.type == "function_call"),
    None,
)

if tool_call is None:
    raise RuntimeError(f"No tool call returned: {response.output}")

print("Tool:", tool_call.name)
print("Raw arguments:", tool_call.arguments)


# 3. Parse the JSON and execute your Python function.
arguments = json.loads(tool_call.arguments)
print("Parsed arguments:", arguments)

page_results = get_page(**arguments)
print("Page characters:", len(page_results))
print("Page preview:", page_results[:300])


# 4. Preserve the model response and attach the function's result.
messages.extend(response.output)
messages.append({
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": page_results,
})


# 5. Ask the LLM to answer using the fetched content.
final_response = openai_client.responses.create(
    model="gpt-6-luna",
    input=messages,
    tools=[get_page_tool],
    tool_choice="none",  # Answer using the content already fetched.
)

print(final_response.output_text)
print("First call input tokens:", response.usage.input_tokens)
print("Second call input tokens:", final_response.usage.input_tokens)



Tool: get_page
Raw arguments: {"PAGE_TITLE":"Capybara"}
Parsed arguments: {'PAGE_TITLE': 'Capybara'}
Page characters: 44767
Page preview: {{Short description|Largest species of rodents}}
{{Other uses}}
{{Good article}}
{{pp|small=yes}}
{{Use dmy dates|date=July 2022}}
{{Speciesbox
| status = LC
| status_system = IUCN3.1
| status_ref = <ref name="iucn">{{cite iucn |author=Reid, F. |year=2016 |title=''Hydrochoerus hydrochaeris'' |articl
The page is about the capybara, the world’s largest living rodent. Native to most of South America, it is a social, semiaquatic herbivore that lives near freshwater and feeds mainly on grasses and aquatic plants. The article also covers its anatomy, behavior, habitat, reproduction, conservation, and place in popular culture.
First call input tokens: 142
Second call input tokens: 13438


In [24]:
print(response.usage.input_tokens)
print(final_response.output_text)

142
The page is about the capybara, the world’s largest living rodent. Native to most of South America, it is a social, semiaquatic herbivore that lives near freshwater and feeds mainly on grasses and aquatic plants. The article also covers its anatomy, behavior, habitat, reproduction, conservation, and place in popular culture.


In [13]:
print(tool_call.name)
print(tool_call.arguments)

get_page
{"PAGE_TITLE":"Capybara"}


In [25]:
tool_call = response.output[0]
tool_call

ResponseFunctionToolCall(arguments='{"PAGE_TITLE":"Capybara"}', call_id='call_18vvLFGeKqHK5hKhgJBu1PWp', name='get_page', type='function_call', id='fc_00fede9e650e0ed5006ad3789c03ec87d2a8552485852e4f66', async_=None, caller=None, namespace=None, status='completed')

In [30]:
print(messages)

[{'role': 'system', 'content': "\nYou're a Wikipedia assistant. \n\nAnswer the user question using the Wikipedia page knowledge base. Provide a concise summary of what the page is about.\n\nUse only facts from the knowledge base when answering.\nIMPORTANT: f you cannot find the answer, inform the user.\n"}, {'role': 'user', 'content': 'What is this page about? https://en.wikipedia.org/wiki/Capybara'}, ResponseFunctionToolCall(arguments='{"PAGE_TITLE":"Capybara"}', call_id='call_18vvLFGeKqHK5hKhgJBu1PWp', name='get_page', type='function_call', id='fc_00fede9e650e0ed5006ad3789c03ec87d2a8552485852e4f66', async_=None, caller=None, namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_18vvLFGeKqHK5hKhgJBu1PWp', 'output': '{{Short description|Largest species of rodents}}\n{{Other uses}}\n{{Good article}}\n{{pp|small=yes}}\n{{Use dmy dates|date=July 2022}}\n{{Speciesbox\n| status = LC\n| status_system = IUCN3.1\n| status_ref = <ref name="iucn">{{cite iucn |aut

In [13]:
# The Agentic Loop

# import json

# Make a query and build the prompt:
question = "What is this page about? https://en.wikipedia.org/wiki/Capybara"

# The make_call Function
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'get_page':
        result = get_page(**arguments)
    else: 
        result = 'not found tool "{name}"'

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

# Update the instructions to test multi-iteration behavior:
instructions = """
You're a Wikipedia assistant. 

Answer the user question using the Wikipedia page knowledge base

Make 2 iterations:

1) in the first iteration, perform one get_page
2) in the second interation, analyze the results from the previous get_page
   and provide a concise summary of what the page is about


Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

"""

# Now define the agentic loop:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-6-luna',
        input=message_history,
        tools=[get_page_tool],
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)   # response.output is a list of output items, including the tool call. extend() adds each item individually to messages

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('Wikipedia ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 1...
executing get_page({"PAGE_TITLE":"Capybara"})...

iteraration number 2...
Wikipedia ASSISTANT: The page is about the **capybara** (*Hydrochoerus hydrochaeris*), the world’s largest living rodent. Native to most of South America, it is a social, semiaquatic herbivore that lives near freshwater and feeds mainly on grasses and aquatic plants. The article also covers its classification, appearance, habitat, behavior, reproduction, conservation, interactions with people, and place in popular culture.



### Question 6. Testing Your Agent - Search Then Fetch
Next, test if your agent can use both tools together.

Ask your agent:

"What are the main threats to capybara populations?"

Your agent should:
- First use search_wikipedia to find pages about capybara threats
- Then use get_page to fetch the relevant pages and extract specific threat information
How many total tool calls (search + fetch combined) did your agent make?

In your answer, provide:

Which tools it used and in what order
The final answer your agent gave

In [15]:
# Adding More Tools
import requests

def search_wikipedia(query):
    response = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "query",
            "format": "json",
            "list": "search",
            "srsearch": query,
        },
        headers={
            "User-Agent": "AgentFlowLearning/1.0 (educational notebook)"
        },
        timeout=20,
    )
    response.raise_for_status()
    return response.json()["query"]["search"]

search_wikipedia_tool = {
    "type": "function",
    "name": "search_wikipedia",
    "description": "Takes a query string as input, calls the Wikipedia Search API, and returns the search results.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the Wikipedia Search API. Use `+` for spaces (for example, lesser+capybara)"
            }
        },
        "required": [
            "query"
        ]
    }
}

# Update make_call to handle the new tool:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'get_page':
        result = get_page(**arguments)
    elif name == 'search_wikipedia':
        result = search_wikipedia(**arguments)
    else: 
        result = 'not found tool "{name}"'

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [20]:
# Multiple Tools Working Together

instructions = """
You're a Wikipedia assistant. 

Answer the user question using the Wikipedia page knowledge base

Make 3 iterations:

1) in the first iteration, perform one search_wikipedia to find pages related to the user question
2) in the second interation, analyze the results from the previous search
   and perform 5 get_page to fetch the top 5 relevant pages and extract specific information
3) synthesise the results into the output, provide a concise summary

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

"""

question = "What are the main threats to capybara populations?"

message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]
tools = [get_page_tool, search_wikipedia_tool]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-6-luna',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('Wikipedia ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 1...
executing search_wikipedia({"query":"capybara threats population"})...

iteraration number 2...
executing get_page({"PAGE_TITLE":"Capybara"})...
executing get_page({"PAGE_TITLE":"Rodent"})...
executing get_page({"PAGE_TITLE":"Jaguar"})...
executing get_page({"PAGE_TITLE":"Black caiman"})...
executing get_page({"PAGE_TITLE":"Yacare caiman"})...

iteraration number 3...
Wikipedia ASSISTANT: The main threats are **human-related**:

- **Hunting** for meat and hides (and, in some places, grease from their skin), which has reduced numbers locally.
- **Killing by people** who view capybaras’ grazing as competition for livestock.

Capybaras are not considered threatened overall, and their populations are stable across most of their South American range. Natural predators—including jaguars, caimans, and anacondas—also cause substantial mortality in the wild. ([Capybara](https://en.wikipedia.org/wiki/Capybara))



### 3 get_page fetch results 
iteraration number 1...
executing search_wikipedia({"query":"capybara threats population"})...

iteraration number 2...
executing get_page({"PAGE_TITLE":"Capybara"})...
executing get_page({"PAGE_TITLE":"Rodent"})...
executing get_page({"PAGE_TITLE":"Jaguar"})...

iteraration number 3...
Wikipedia ASSISTANT: Wikipedia identifies **human pressure** as the main threat: capybaras are hunted for their meat, hides and fat, and are sometimes killed by people who see their grazing as competition with livestock. Hunting has reduced their numbers in some areas. Their populations are considered stable across most of their range, and the species is not currently considered threatened. Natural predators—including jaguars, caimans, eagles and anacondas—also take a toll, especially on young capybaras. — [Capybara](https://en.wikipedia.org/wiki/Capybara)

iteraration number 1...
executing search_wikipedia({"query":"capybara threats population"})...

iteraration number 2...
executing get_page({"PAGE_TITLE":"Capybara"})...
executing get_page({"PAGE_TITLE":"Rodent"})...
executing get_page({"PAGE_TITLE":"Jaguar"})...
executing get_page({"PAGE_TITLE":"Black caiman"})...
executing get_page({"PAGE_TITLE":"Yacare caiman"})...

iteraration number 3...
Wikipedia ASSISTANT: The main threats are **human-related**:

- **Hunting** for meat and hides (and, in some places, grease from their skin), which has reduced numbers locally.
- **Killing by people** who view capybaras’ grazing as competition for livestock.

Capybaras are not considered threatened overall, and their populations are stable across most of their South American range. Natural predators—including jaguars, caimans, and anacondas—also cause substantial mortality in the wild. ([Capybara](https://en.wikipedia.org/wiki/Capybara))

In [ ]:
#  import the Agent class:
from pydantic_ai import Agent

# Create the agent:
search_agent = Agent(
    name='search',
    model='openai:gpt-6-luna',
    instructions=instructions,
    tools=tools
)

In [ ]:
# 

